# 生成 hey hi 困难负样本音频

为 Hey Kamana 模型生成 hey hi 负样本 WAV 音频，可在 Colab GPU 或本地 Jupyter 环境中按顺序运行。

默认生成快、正常、慢三档共 3000 条原始音频，先试听，再批量生成、去重，并逐档按 80%/10%/10% 划分。
最终 ZIP 只包含 training/、validation/、testing/ 下的 WAV 文件，保留原始采样率和完整发音。
生成参数与音频清单仅保存在本地数据目录中，方便检查与复用。

In [1]:
# 1. 定位仓库；Colab 中没有 checkout 时自动克隆。
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Seeker32/micro-wake-word.git"
cwd = Path.cwd()
candidates = (cwd, cwd.parent, cwd / "micro-wake-word", Path("/content/micro-wake-word"))
repo_path = next((p.resolve() for p in candidates if (p / "microwakeword/audio/augmentation.py").is_file()), None)
if repo_path is None:
    import importlib.util
    if not Path("/content").is_dir():
        raise FileNotFoundError("请从仓库或 notebooks 目录启动，或在 candidates 中添加本地仓库路径。")
    repo_path = Path("/content/micro-wake-word")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_path)], check=True)
os.chdir(repo_path)
print("仓库:", repo_path)
print("当前 kernel:", sys.executable)

仓库: /home/seeker32/projects/micro-wake-word
当前 kernel: /home/seeker32/projects/micro-wake-word/.venv/bin/python


In [2]:
# 2. 集中参数。修改生成参数时，请更换 DATASET_NAME，避免混用旧数据。
DATASET_NAME = "hey_hi_negative_v1"
PHRASE = "hey hi"
PHONEMES = "hˈeɪ hˈaɪ"  # 两个完整音节；先试听确认读音
PROFILES = [
    {"name": "fast", "length_scale": 0.65, "samples": 1000},
    {"name": "normal", "length_scale": 1.0, "samples": 1000},
    {"name": "slow", "length_scale": 1.25, "samples": 1000},
]
BATCH_SIZE = 50  # 显存不足时调低；CPU 可设为 1
MAX_SPEAKERS = 100
NOISE_SCALES = [0.667, 0.8, 1.0]
NOISE_SCALE_WS = [0.7, 0.9]
SPLIT_SEED = 42

dataset_root = repo_path / "generated_datasets" / DATASET_NAME
raw_root = dataset_root / "raw"
piper_dir = repo_path / "piper-sample-generator"
model_path = piper_dir / "models" / "en_US-libritts_r-medium.pt"
MODEL_URL = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"

assert len({p["name"] for p in PROFILES}) == len(PROFILES)
assert all(p["samples"] >= 10 and p["length_scale"] > 0 for p in PROFILES)
print("数据集:", dataset_root)
print("原始样本总数:", sum(p["samples"] for p in PROFILES))

数据集: /home/seeker32/projects/micro-wake-word/generated_datasets/hey_hi_negative_v1
原始样本总数: 3000


安装单元格只准备 Piper 合成所需的依赖。已有 PyTorch 时复用当前版本，避免覆盖本地 CUDA 配置；
没有时再安装。Piper 从源码目录运行。
若 notebook 提示重启 kernel，重启后重新运行路径、参数单元格，再继续。

In [3]:
# 3. 安装、准备 Piper 和模型；下载失败不会留下可被误用的完整模型文件。
import importlib.util
import urllib.request

def run(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)

if importlib.util.find_spec("pip") is None:
    run([sys.executable, "-m", "ensurepip", "--upgrade"])
if importlib.util.find_spec("torch") is None:
    run([sys.executable, "-m", "pip", "install", "torch>=2,<3"])
run([sys.executable, "-m", "pip", "install", "numpy>=2,<3", "piper-tts==1.3.0"])

if not piper_dir.exists():
    run(["git", "clone", "--depth", "1",
         "https://github.com/rhasspy/piper-sample-generator", str(piper_dir)])
if not (piper_dir / "piper_sample_generator" / "__main__.py").is_file():
    raise FileNotFoundError(f"Piper checkout 不完整: {piper_dir}")
if not model_path.is_file():
    model_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = model_path.with_suffix(".pt.download")
    urllib.request.urlretrieve(MODEL_URL, temporary)
    temporary.replace(model_path)
if not Path(str(model_path) + ".json").is_file():
    raise FileNotFoundError(f"缺少模型配置: {model_path}.json；请检查 Piper checkout。")


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip


In [4]:
# 4. 生成函数：每档独立指定语速，避免批次参数循环导致快语速覆盖不足。
import hashlib
import json
import wave

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

model_sha256 = sha256(model_path)
piper_commit = subprocess.check_output(
    ["git", "-C", str(piper_dir), "rev-parse", "HEAD"], text=True
).strip()

def synthesize(profile, count, output_dir):
    settings = {
        "phrase": PHRASE, "phonemes": PHONEMES,
        "length_scale": profile["length_scale"], "samples": count,
        "batch_size": min(BATCH_SIZE, count), "max_speakers": MAX_SPEAKERS,
        "noise_scales": NOISE_SCALES, "noise_scale_ws": NOISE_SCALE_WS,
        "model_sha256": model_sha256, "piper_commit": piper_commit,
    }
    marker = output_dir / "generation.json"
    if output_dir.exists():
        if (not marker.is_file() or json.loads(marker.read_text()) != settings
                or len(list(output_dir.glob("*.wav"))) != count):
            raise FileExistsError(f"{output_dir} 已存在但未完成或参数不同。请更换 DATASET_NAME，或手动检查该目录。")
        print("复用已完成的生成:", output_dir)
        return
    command = [
        sys.executable, "-X", "faulthandler", "-m", "piper_sample_generator", PHONEMES,
        "--phoneme-input", "--model", str(model_path),
        "--max-samples", str(count), "--batch-size", str(min(BATCH_SIZE, count)),
        "--max-speakers", str(MAX_SPEAKERS),
        "--length-scales", str(profile["length_scale"]),
        "--noise-scales", *map(str, NOISE_SCALES),
        "--noise-scale-ws", *map(str, NOISE_SCALE_WS),
        "--output-dir", str(output_dir),
    ]
    run(command, cwd=piper_dir)
    paths = sorted(output_dir.glob("*.wav"))
    if len(paths) != count:
        raise RuntimeError(f"{output_dir}: 预期 {count} 条，实际 {len(paths)} 条")
    for path in paths:
        with wave.open(str(path), "rb") as wav:
            if wav.getnframes() == 0 or wav.getnchannels() != 1 or wav.getsampwidth() != 2:
                raise ValueError(f"空音频或不是单声道 PCM16: {path}")
    marker.write_text(json.dumps(settings, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

In [5]:
# 5. 每档先试听 1 条；确认尤其是 fast 档仍为完整 hey hi，再运行批量生成。
from IPython.display import Audio, display

for profile in PROFILES:
    preview = dataset_root / "preview" / profile["name"]
    synthesize(profile, 1, preview)
    print(profile["name"], "length_scale =", profile["length_scale"])
    display(Audio(filename=str(preview / "0.wav")))

DEBUG:__main__:Loading /home/seeker32/projects/micro-wake-word/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/1 complete
INFO:__main__:Done


fast length_scale = 0.65


DEBUG:__main__:Loading /home/seeker32/projects/micro-wake-word/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/1 complete
INFO:__main__:Done


normal length_scale = 1.0


DEBUG:__main__:Loading /home/seeker32/projects/micro-wake-word/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/1 complete
INFO:__main__:Done


slow length_scale = 1.25


In [6]:
# 6. 批量生成。试听目录不会被加入数据集。
for profile in PROFILES:
    synthesize(profile, profile["samples"], dataset_root / "generated" / profile["name"])

DEBUG:__main__:Loading /home/seeker32/projects/micro-wake-word/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/20 complete
DEBUG:__main__:Batch 2/20 complete
DEBUG:__main__:Batch 3/20 complete
DEBUG:__main__:Batch 4/20 complete
DEBUG:__main__:Batch 5/20 complete
DEBUG:__main__:Batch 6/20 complete
DEBUG:__main__:Batch 7/20 complete
DEBUG:__main__:Batch 8/20 complete
DEBUG:__main__:Batch 9/20 complete
DEBUG:__main__:Batch 10/20 complete
DEBUG:__main__:Batch 11/20 complete
DEBUG:__main__:Batch 12/20 complete
DEBUG:__main__:Batch 13/20 complete
DEBUG:__main__:Batch 14/20 complete
DEBUG:__main__:Batch 15/20 complete
DEBUG:__main__:Batch 16/20 complete
DEBUG:__main__:Batch 17/20 complete
DEBUG:__main__:Batch 18/20 complete
DEBUG:__main__:Batch 19/20 complete
DEBUG:__main__:Batch 20/20 complete
INFO:__main__:Done
DEBUG:__main__:Loading /home/seeker32/projects/micro-wake-word/pi

下面按原始 PCM 音频去重，再逐档按 80%/10%/10% 划分。同一条原始录音只属于一个集合。
输出 raw/<split>/*.wav 保留原始采样率和完整长度，不裁剪、不补静音。
空音频和重复音频记录到本地 rejected.json。

该划分只保证原始录音不重复，不保证说话人隔离。

In [7]:
# 7. 去重、划分并保存可追溯清单。重跑时拒绝覆盖既有划分。
import random
import shutil
from collections import Counter

manifest_path = dataset_root / "manifest.json"
raw_root = dataset_root / "raw"
if manifest_path.exists() or raw_root.exists():
    raise FileExistsError("划分已存在。继续运行后面的单元格；如需重新生成，请更换 DATASET_NAME。")

seen = set()
records, rejected = [], []
rng = random.Random(SPLIT_SEED)
for profile in PROFILES:
    accepted = []
    for path in sorted((dataset_root / "generated" / profile["name"]).glob("*.wav")):
        with wave.open(str(path), "rb") as wav:
            if wav.getnchannels() != 1 or wav.getsampwidth() != 2:
                raise ValueError(f"只接受单声道 PCM16: {path}")
            rate, frames = wav.getframerate(), wav.getnframes()
            pcm = wav.readframes(frames)
        duration = frames / rate
        fingerprint = hashlib.sha256(str(rate).encode() + b":" + pcm).hexdigest()
        reason = ("empty_audio" if not frames else
                  "duplicate_pcm" if fingerprint in seen else None)
        if reason:
            rejected.append({"path": str(path.relative_to(dataset_root)), "reason": reason, "duration_s": duration})
            continue
        seen.add(fingerprint)
        accepted.append({
            "source": str(path.relative_to(dataset_root)), "sha256": sha256(path),
            "pcm_sha256": fingerprint, "profile": profile["name"],
            "sample_rate": rate, "duration_s": duration, "truth": False,
        })
    if len(accepted) < 10:
        raise ValueError(f"{profile['name']} 只有 {len(accepted)} 条可用音频，请检查合成结果。")
    rng.shuffle(accepted)
    held_out = max(1, len(accepted) // 10)
    for index, row in enumerate(accepted):
        row["split"] = ("validation" if index < held_out else
                        "testing" if index < 2 * held_out else "training")
        records.append(row)

# 所有档位检查通过后再写入 raw，避免校验失败时留下不完整划分。
for row in records:
    source = dataset_root / row["source"]
    target = raw_root / row["split"] / f"{row['profile']}_{source.name}"
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
    row["path"] = str(target.relative_to(dataset_root))

manifest = {
    "phrase": PHRASE, "phonemes": PHONEMES, "truth": False, "split_seed": SPLIT_SEED,
    "split_policy": "PCM-deduplicated source recordings, stratified by speed; not speaker-disjoint",
    "model_url": MODEL_URL, "model_sha256": model_sha256, "piper_commit": piper_commit,
    "files": records,
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(dataset_root / "rejected.json").write_text(json.dumps(rejected, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("接受:", Counter((r["split"], r["profile"]) for r in records))
print("排除:", Counter(r["reason"] for r in rejected))

接受: Counter({('training', 'fast'): 800, ('training', 'normal'): 800, ('training', 'slow'): 800, ('validation', 'fast'): 100, ('testing', 'fast'): 100, ('validation', 'normal'): 100, ('testing', 'normal'): 100, ('validation', 'slow'): 100, ('testing', 'slow'): 100})
排除: Counter()


In [ ]:
# 8. 重启后先运行步骤 1–4，再从这里加载清单，跳过生成与划分。
from collections import Counter
manifest = json.loads((dataset_root / "manifest.json").read_text(encoding="utf-8"))
records = manifest["files"]
assert manifest["truth"] is False and all(r["truth"] is False for r in records)
assert len({r["pcm_sha256"] for r in records}) == len(records), "存在跨集合重复原始音频"
for row in records:
    if sha256(dataset_root / row["path"]) != row["sha256"]:
        raise ValueError(f"音频已改变: {row['path']}")
print("原始录音:", Counter(r["split"] for r in records))

In [8]:
# 9. 只打包划分后的 WAV 音频；ZIP 内不包含模型、清单或试听文件。
import zipfile

archive_path = repo_path / f"{DATASET_NAME}_audio.zip"
audio_paths = sorted(raw_root.rglob("*.wav"))
if len(audio_paths) != len(records):
    raise ValueError("WAV 数量与清单不一致，请先检查音频划分。")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in audio_paths:
        archive.write(path, arcname=str(path.relative_to(raw_root)))
print("音频压缩包:", archive_path)
print("WAV 数量:", len(audio_paths))
print("大小 MiB:", round(archive_path.stat().st_size / 1024**2, 2))

DOWNLOAD_IN_COLAB = False  # 改为 True 后运行，即可触发浏览器下载
if DOWNLOAD_IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))


音频压缩包: /home/seeker32/projects/micro-wake-word/hey_hi_negative_v1_audio.zip
WAV 数量: 3000
大小 MiB: 72.57
